<a href="https://colab.research.google.com/github/thao-cpu/Deepfake-eKYC-SiameseResNet18/blob/feature%2Fmodel/modelDeepfake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cài đặt thư viện pytorch-metric-learning dùng cho Batch Hard Mining và kiểm tra môi trường
!pip install pytorch-metric-learning -q

import torch
import os
from google.colab import drive
drive.mount("/content/drive")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Không có GPU")

# Giải nén dataset nếu chưa có
TAR_FILE = "/content/drive/MyDrive/Deepfake/output_dataset.tar"
DATASET_DIR = "/content/output_dataset"
required_dirs = ["train/real", "train/fake", "val/real", "val/fake", "test/real", "test/fake"]

if not all(os.path.isdir(os.path.join(DATASET_DIR, d)) for d in required_dirs):
    print("Đang giải nén dataset...")
    !tar -xf "$TAR_FILE" -C /content
    print("Dataset sẵn sàng!")
else:
    print("Dataset đã sẵn sàng!")

Mounted at /content/drive
PyTorch: 2.11.0+cu128
GPU: Tesla T4
Đang giải nén dataset...
Dataset sẵn sàng!


In [ ]:
%%writefile /content/data_analysis.py
import os
import pandas as pd
import matplotlib.pyplot as plt

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def extract_video_id(filename):
    # Tách ID video gốc từ tên file ảnh (ví dụ: video123_f10.jpg -> video123)
    stem = os.path.splitext(filename)[0]
    if "_f" in stem:
        return stem.split("_f")[0]
    return stem

def generate_metadata_and_eda(dataset_root):
    data = []
    # Quét qua tất cả thư mục train/val/test và real/fake
    for split in ["train", "val", "test"]:
        for label in ["real", "fake"]:
            class_dir = os.path.join(dataset_root, split, label)
            for filename in os.listdir(class_dir):
                if not filename.lower().endswith(IMAGE_EXTENSIONS): continue
                data.append({
                    "image_path": os.path.join(class_dir, filename),
                    "split": split,
                    "label": label,
                    "video_id": extract_video_id(filename),
                    "filename": filename
                })

    df = pd.DataFrame(data)
    csv_path = os.path.join(dataset_root, "metadata.csv")
    df.to_csv(csv_path, index=False)
    print(f"Metadata saved: {csv_path} | Total images: {len(df)}")

    # Kiểm tra data leakage (không có video nào xuất hiện ở cả train và test)
    train_vids = set(df[df["split"] == "train"]["video_id"])
    val_vids = set(df[df["split"] == "val"]["video_id"])
    test_vids = set(df[df["split"] == "test"]["video_id"])
    if not (train_vids & val_vids) and not (train_vids & test_vids):
        print("OK: No video-level leakage detected")
    else:
        print("WARNING: Data leakage detected!")

    # Vẽ biểu đồ phân bố Real/Fake
    stats = df.groupby(["split", "label"]).size().unstack(fill_value=0)
    stats.plot(kind="bar", figsize=(8, 6))
    plt.title("Real/Fake Distribution")
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    generate_metadata_and_eda("/content/output_dataset")

Writing /content/data_analysis.py


In [ ]:
!python /content/data_analysis.py

Metadata saved: /content/output_dataset/metadata.csv | Total images: 40642
OK: No video-level leakage detected
Figure(800x600)


In [ ]:
%%writefile /content/dataset.py
import os
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import collections
import random
import numpy as np

class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_frames_per_video=5):
        self.transform = transform
        self.class_to_idx = {"real": 0, "fake": 1}
        self.samples = []

        for class_name, label in self.class_to_idx.items():
            class_dir = Path(root_dir) / class_name
            video_frames = collections.defaultdict(list)

            # Đọc ảnh và nhóm theo video_id
            for image_path in class_dir.iterdir():
                if image_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                    stem = image_path.stem
                    video_id = stem.split("_f")[0] if "_f" in stem else stem
                    video_frames[video_id].append(image_path)

            # CHỐNG OVERFITTING: Chỉ lấy ngẫu nhiên tối đa 5 frame/video
            for video_id, frames in video_frames.items():
                if len(frames) > max_frames_per_video:
                    selected_frames = random.sample(frames, max_frames_per_video)
                else:
                    selected_frames = frames
                for image_path in selected_frames:
                    self.samples.append((str(image_path), label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

def get_train_transform():
    return transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.1))
    ])

def get_val_test_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def create_dataloader(root_dir, batch_size=32, shuffle=False, num_workers=2, transform=None):
    dataset = DeepfakeDataset(root_dir=root_dir, transform=transform)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=torch.cuda.is_available(), worker_init_fn=seed_worker)

Writing /content/dataset.py


In [ ]:
%%writefile /content/model.py
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

class ResNet18Baseline(nn.Module):
    def __init__(self, pretrained=True, dropout=0.3, freeze_backbone=False):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        if freeze_backbone:
            for param in self.backbone.parameters(): param.requires_grad = False
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, 1))

    def forward(self, x): return self.backbone(x).squeeze(1)

class SiameseResNet18(nn.Module):
    def __init__(self, pretrained=True, embedding_dim=128, dropout=0.3, freeze_backbone=False):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        if freeze_backbone:
            for param in self.backbone.parameters(): param.requires_grad = False
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.embedding_head = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        # Thêm Classification Head để dự đoán trực tiếp (Joint Loss)
        self.classifier = nn.Linear(embedding_dim, 1)

    def encode(self, x):
        features = self.backbone(x)
        embedding = self.embedding_head(features)
        # L2 Normalize: Ép vector embedding nằm trên mặt cầu đơn vị, giúp khoảng cách ổn định
        return F.normalize(embedding, p=2, dim=1)

    def forward(self, x):
        emb = self.encode(x)
        logits = self.classifier(emb).squeeze(1)
        return emb, logits

Writing /content/model.py


In [ ]:
%%writefile /content/train_baseline.py
import os, torch, random, numpy as np
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import f1_score
from dataset import create_dataloader, get_train_transform, get_val_test_transform
from model import ResNet18Baseline

DATASET_DIR = "/content/output_dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, all_labels, all_preds = 0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss = criterion(logits, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * images.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend((torch.sigmoid(logits) >= 0.5).float().cpu().numpy())
    return total_loss / len(loader.dataset), f1_score(all_labels, all_preds, zero_division=0)

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, all_labels, all_preds = 0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend((torch.sigmoid(logits) >= 0.5).float().cpu().numpy())
    return total_loss / len(loader.dataset), f1_score(all_labels, all_preds, zero_division=0)

def main():
    print("Device:", DEVICE)
    train_loader = create_dataloader(f"{DATASET_DIR}/train", batch_size=32, shuffle=True, transform=get_train_transform())
    val_loader = create_dataloader(f"{DATASET_DIR}/val", batch_size=32, transform=get_val_test_transform())

    # Khởi tạo 1 lần duy nhất
    model = ResNet18Baseline(pretrained=True, dropout=0.3, freeze_backbone=False).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    best_model_path = "/content/drive/MyDrive/Deepfake/best_baseline.pth"
    best_val_f1 = -1
    if os.path.exists(best_model_path):
        best_val_f1 = torch.load(best_model_path, map_location="cpu").get("val_f1", -1)
        print(f"Existing Best Val F1: {best_val_f1:.4f}")

    for epoch in range(10):
        train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1 = validate(model, val_loader, criterion)
        print(f"Epoch [{epoch+1}/10] Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict(), "val_f1": best_val_f1}, best_model_path)
            print(f"✓ Saved best model (Val F1={best_val_f1:.4f})")

if __name__ == "__main__":
    main()

Writing /content/train_baseline.py


In [ ]:
!python /content/train_baseline.py

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 191MB/s]
Existing Best Val F1: 0.5354
Epoch [1/10] Train Loss: 0.6610 | Train F1: 0.5971 | Val Loss: 0.7011 | Val F1: 0.5346
Epoch [2/10] Train Loss: 0.5846 | Train F1: 0.6852 | Val Loss: 0.7606 | Val F1: 0.5078
Epoch [3/10] Train Loss: 0.5192 | Train F1: 0.7419 | Val Loss: 0.7735 | Val F1: 0.5696
✓ Saved best model (Val F1=0.5696)
Epoch [4/10] Train Loss: 0.4549 | Train F1: 0.7802 | Val Loss: 0.8061 | Val F1: 0.5283
Epoch [5/10] Train Loss: 0.4117 | Train F1: 0.8082 | Val Loss: 1.0041 | Val F1: 0.4871
Epoch [6/10] Train Loss: 0.3994 | Train F1: 0.8119 | Val Loss: 0.9215 | Val F1: 0.5036
Epoch [7/10] Train Loss: 0.3325 | Train F1: 0.8477 | Val Loss: 0.9484 | Val F1: 0.5566
Epoch [8/10] Train Loss: 0.3255 | Train F1: 0.8549 | Val Loss: 1.2280 | Val F1: 0.4379
Epoch [9/10] Train Loss: 0.2889 | Train F1: 0.8654 | V

In [ ]:
# load baseline ResNet18 tốt nhất đã train, chạy trên test set và đánh giá hiệu năng bằng Acc, Precision, Recall, F1-score, Classification Report và Confusion Matrix
%%writefile /content/evaluate_baseline.py
import torch
import matplotlib.pyplot as plt
import os
import random
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
from dataset import (
    create_dataloader,
    get_val_test_transform
)
from model import ResNet18Baseline
DATASET_DIR = "/content/output_dataset"
CHECKPOINT = "/content/drive/MyDrive/Deepfake/best_baseline.pth"
SAVE_DIR = ("/content/drive/MyDrive/Deepfake/" "evaluation_baseline")
os.makedirs(
    SAVE_DIR,
    exist_ok=True
)
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
def main():
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
  # lấy dữ liệu từ output_dataset/test và không augmentation vì đây là test set
    test_loader = create_dataloader(
        f"{DATASET_DIR}/test",
        batch_size=32,
        shuffle=False,
        num_workers=2,
        transform=get_val_test_transform()
    )
    model = ResNet18Baseline(
        pretrained=False, # hợp lý vì không cần tải lại ImageNet weights
        dropout=0.3
    ).to(DEVICE)
    checkpoint = torch.load(
        CHECKPOINT,
        map_location=DEVICE
    ) # load toàn bộ weights đã train
    model.load_state_dict(
        checkpoint["model_state_dict"]
    ) # [cấu trúc ResNet18 + best_baseline.pth -> Trained Baseline]
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad(): # không tính gradient vì đây là evaluation
        for images, labels in test_loader:
            images = images.to(
                DEVICE
            )
            logits = model(images) # model tạo logit
            probabilities = torch.sigmoid(
                logits
            ) # chuyển logit thành xác suất
            predictions = (
                probabilities >= 0.5
            ).long() # dùng threshold 0.5
            y_true.extend(
                labels.long().numpy()
            )
            y_pred.extend(
                predictions.cpu().numpy()
            )
    accuracy = accuracy_score(
        y_true,
        y_pred
    ) # tỷ lệ dự đoán đúng trên tổng số mẫu
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    ) # trong những ảnh model dự đoán là fake, có bao nhiêu ảnh thực sự fake
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    ) # trong tất cả ảnh fake thật, model phát hiện được bao nhiêu
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    ) # cân bằng Precision và Recall
    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
    print("\n" + "=" * 50)
    print("BASELINE TEST RESULTS")
    print("=" * 50)
    print(
        f"Accuracy : {accuracy:.4f}"
    )
    print(
        f"Precision: {precision:.4f}"
    )
    print(
        f"Recall   : {recall:.4f}"
    )
    print(
        f"F1-score : {f1:.4f}"
    )
    print(
        f"Macro F1 : {macro_f1:.4f}"
    )
    print("\nClassification Report:")
    report = classification_report(
        y_true,
        y_pred,
        target_names=[
            "real",
            "fake"
        ],
        zero_division=0
    )
    print(report) # bảng chi tiết theo từng class, giúp xem model có đang tốt ở cả real và fake hay chỉ tốt ở một class
    cm = confusion_matrix(
        y_true,
        y_pred
    ) # ma trận nhầm lẫn:
    # TN (real -> dự đoán real)
    # FP (real -> dự đoán fake)
    # FN (fake -> dự đoán real)
    # TP (fake -> dự đoán fake)
    print("Confusion Matrix:")
    print(cm)
    # vẽ thành biểu đồ trực quan
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "real",
            "fake"
        ]
    ).plot(values_format="d")
    plt.title(
        "Baseline ResNet18"
    )
    plt.tight_layout()
    confusion_path = os.path.join(
        SAVE_DIR,
        "baseline_confusion_matrix.png"
    )
    plt.savefig(
        confusion_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    # SAVE TEXT RESULTS TO DRIVE
    result_path = os.path.join(SAVE_DIR,"baseline_test_results.txt")
    with open(result_path,"w",encoding="utf-8") as f:
        f.write("BASELINE RESNET18 TEST RESULTS\n")
        f.write("=" * 50 + "\n")
        f.write(f"Checkpoint epoch: "f"{checkpoint['epoch']}\n")
        f.write(f"Best Validation F1: "f"{checkpoint['val_f1']:.4f}\n\n")
        f.write(f"Accuracy : {accuracy:.4f}\n")
        f.write(f"Precision: {precision:.4f}\n")
        f.write(f"Recall   : {recall:.4f}\n")
        f.write(f"F1-score : {f1:.4f}\n")
        f.write(f"Macro F1 : {macro_f1:.4f}\n")
        f.write("\nClassification Report:\n")
        f.write(report)
        f.write("\nConfusion Matrix:\n")
        f.write(str(cm))
    # FINISH
    print("\n" + "=" * 60)
    print("Baseline evaluation completed!")
    print("Results saved to:")
    print(SAVE_DIR)
    print("\nText result:")
    print(result_path)
    print("\nConfusion matrix:")
    print(confusion_path)
if __name__ == "__main__":
    main()
# dùng best ResNet18 Baseline checkpoint đã lưu
# để dự đoán trên test/, sau đó tính
# Accuracy, Precision, Recall, F1-score,
# in Classification Report và vẽ Confusion Matrix.

Writing /content/evaluate_baseline.py


In [ ]:
!python /content/evaluate_baseline.py


BASELINE TEST RESULTS
Accuracy : 0.6049
Precision: 0.5896
Recall   : 0.6362
F1-score : 0.6120
Macro F1 : 0.6048

Classification Report:
              precision    recall  f1-score   support

        real       0.62      0.57      0.60       501
        fake       0.59      0.64      0.61       481

    accuracy                           0.60       982
   macro avg       0.61      0.61      0.60       982
weighted avg       0.61      0.60      0.60       982

Confusion Matrix:
[[288 213]
 [175 306]]
Figure(640x480)

Baseline evaluation completed!
Results saved to:
/content/drive/MyDrive/Deepfake/evaluation_baseline

Text result:
/content/drive/MyDrive/Deepfake/evaluation_baseline/baseline_test_results.txt

Confusion matrix:
/content/drive/MyDrive/Deepfake/evaluation_baseline/baseline_confusion_matrix.png


In [ ]:
%%writefile /content/train_siamese.py
import torch, os, random, numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from dataset import DeepfakeDataset, get_train_transform, get_val_test_transform, seed_worker
from model import SiameseResNet18
from sklearn.metrics import f1_score
from pytorch_metric_learning import losses, miners

DATASET_DIR = "/content/output_dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 20
LR = 5e-5           # TĂNG LR LÊN 5e-5 CHO NÓ HỌC NHANH HƠN
WEIGHT_DECAY = 1e-4 # GIẢM WEIGHT DECAY XUỐNG 1e-4

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

def run_epoch(model, loader, miner, loss_triplet, loss_bce, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_samples, all_labels, all_preds = 0, 0, [], []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if is_train: optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            embeddings, logits = model(images)
            hard_pairs = miner(embeddings, labels.long())

            # THAY ĐỔI TRỌNG TÂM: Đánh giá trọng số Joint Loss
            # BCE Loss (Phân loại) được nhân 1.0, Triplet Loss (Khoảng cách) nhân 0.5
            # Giúp model ưu tiên phân loại đúng, dùng khoảng cách làm phụ
            l_triplet = loss_triplet(embeddings, labels.long(), hard_pairs)
            l_bce = loss_bce(logits, labels.float())
            loss = l_bce + 0.5 * l_triplet

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend((torch.sigmoid(logits) >= 0.5).float().cpu().numpy())

    return total_loss / total_samples, f1_score(all_labels, all_preds, zero_division=0, average="macro")

def main():
    print("Device:", DEVICE)
    train_loader = DataLoader(DeepfakeDataset(f"{DATASET_DIR}/train", transform=get_train_transform()), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, worker_init_fn=seed_worker)
    val_loader = DataLoader(DeepfakeDataset(f"{DATASET_DIR}/val", transform=get_val_test_transform()), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, worker_init_fn=seed_worker)

    # GIẢM DROPOUT XUỐNG 0.4 (0.5 hơi nặng, làm model quên mất đặc trưng phân loại)
    model = SiameseResNet18(pretrained=True, embedding_dim=128, dropout=0.4, freeze_backbone=False).to(DEVICE)
    miner = miners.BatchHardMiner()
    loss_triplet = losses.TripletMarginLoss(margin=0.3)
    loss_bce = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # ĐỔI SCHEDULER: CosineAnnealingLR giúp LR giảm dần về 0 mượt mà
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_val_f1 = -1
    save_path = "/content/drive/MyDrive/Deepfake/best_siamese.pth"
    if os.path.exists(save_path):
        best_val_f1 = torch.load(save_path, map_location="cpu").get("val_f1", -1)
        print(f"Existing Best Val F1: {best_val_f1:.4f}")

    for epoch in range(EPOCHS):
        train_loss, train_f1 = run_epoch(model, train_loader, miner, loss_triplet, loss_bce, optimizer)
        val_loss, val_f1 = run_epoch(model, val_loader, miner, loss_triplet, loss_bce)

        # Cập nhật Cosine Scheduler
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | LR: {current_lr:.2e}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict(), "val_f1": best_val_f1}, save_path)
            print(f"✓ Best Siamese saved (Val F1={val_f1:.4f})")

if __name__ == "__main__":
    main()

Writing /content/train_siamese.py


In [ ]:
!python /content/train_siamese.py

Device: cuda
Existing Best Val F1: 0.5206
Epoch [1/20] Train Loss: 0.9428 | Train F1: 0.4231 | Val Loss: 0.6989 | Val F1: 0.4955 | LR: 4.97e-05
Epoch [2/20] Train Loss: 0.9260 | Train F1: 0.5483 | Val Loss: 0.6932 | Val F1: 0.5780 | LR: 4.88e-05
✓ Best Siamese saved (Val F1=0.5780)
Epoch [3/20] Train Loss: 0.9182 | Train F1: 0.6383 | Val Loss: 0.6897 | Val F1: 0.5905 | LR: 4.73e-05
✓ Best Siamese saved (Val F1=0.5905)
Epoch [4/20] Train Loss: 0.9110 | Train F1: 0.6799 | Val Loss: 0.6869 | Val F1: 0.5937 | LR: 4.53e-05
✓ Best Siamese saved (Val F1=0.5937)
Epoch [5/20] Train Loss: 0.9030 | Train F1: 0.7127 | Val Loss: 0.6860 | Val F1: 0.6167 | LR: 4.28e-05
✓ Best Siamese saved (Val F1=0.6167)
Epoch [6/20] Train Loss: 0.8945 | Train F1: 0.7493 | Val Loss: 0.6860 | Val F1: 0.5675 | LR: 3.99e-05
Epoch [7/20] Train Loss: 0.8868 | Train F1: 0.7742 | Val Loss: 0.6840 | Val F1: 0.5950 | LR: 3.66e-05
Epoch [8/20] Train Loss: 0.8722 | Train F1: 0.8073 | Val Loss: 0.6859 | Val F1: 0.5636 | LR: 3.3

In [ ]:
%%writefile /content/evaluate_siamese.py
import numpy as np
import os
import torch
import random
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from dataset import DeepfakeDataset, get_val_test_transform
from model import SiameseResNet18

DATASET_ROOT = "/content/output_dataset"
CHECKPOINT = "/content/drive/MyDrive/Deepfake/best_siamese.pth"
SAVE_DIR = "/content/drive/MyDrive/Deepfake/evaluation_siamese"
os.makedirs(SAVE_DIR, exist_ok=True)
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def main():
    print("Device:", DEVICE)
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    model = SiameseResNet18(pretrained=False, embedding_dim=128, dropout=0.3).to(DEVICE)
    checkpoint = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print("Loaded epoch:", checkpoint["epoch"])
    print("Val F1:", checkpoint.get("val_f1", "N/A"))

    test_dataset = DeepfakeDataset(root_dir=f"{DATASET_ROOT}/test", transform=get_val_test_transform())
    from dataset import seed_worker
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, worker_init_fn=seed_worker)
    print("Test samples:", len(test_dataset))

    y_true = []
    y_prob = [] # Lưu lại xác suất để tìm threshold

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE, non_blocking=True)
            embeddings = model.encode(images)
            logits = model.classifier(embeddings).squeeze(1)
            probabilities = torch.sigmoid(logits)

            y_true.extend(labels.long().numpy())
            y_prob.extend(probabilities.cpu().numpy())

    # Tự động tìm ngưỡng Threshold tốt nhất dựa trên F1-SCORE
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (np.array(y_prob) >= thresh).astype(int)
        # SỬA Ở ĐÂY: BỎ average="macro" đi, để nó mặc định tính F1 cho class Fake
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    print(f"\nBest threshold found: {best_thresh:.2f} with Macro F1: {best_f1:.4f}")

    # Dùng threshold tốt nhất để dự đoán
    y_pred = (np.array(y_prob) >= best_thresh).astype(int)

    # Tính toán Metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print("\n" + "=" * 60)
    print("SIAMESE RESNET18 TEST RESULTS")
    print("=" * 60)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")

    print("\nClassification Report:")
    report = classification_report(y_true, y_pred, target_names=["real", "fake"], digits=4, zero_division=0)
    print(report)

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:")
    print(cm)

    # Vẽ Confusion Matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["real", "fake"]).plot(values_format="d")
    plt.title(f"Siamese ResNet18 - Test Confusion Matrix (Thresh={best_thresh:.2f})")
    plt.tight_layout()
    confusion_path = os.path.join(SAVE_DIR, "siamese_confusion_matrix.png")
    plt.savefig(confusion_path, dpi=300, bbox_inches="tight")
    plt.show()

    # Lưu Predictions
    PREDICTION_PATH = os.path.join(SAVE_DIR, "siamese_predictions.npy")
    np.save(PREDICTION_PATH, y_pred)

    # Lưu Text Results
    result_path = os.path.join(SAVE_DIR, "siamese_test_results.txt")
    with open(result_path, "w", encoding="utf-8") as f:
        f.write("SIAMESE RESNET18 TEST RESULTS\n")
        f.write("=" * 50 + "\n")
        f.write(f"Checkpoint epoch: {checkpoint['epoch']}\n")
        f.write(f"Best Validation F1: {checkpoint.get('val_f1', 'N/A')}\n\n")
        f.write(f"Optimal Threshold: {best_thresh:.2f}\n")
        f.write(f"Accuracy : {accuracy:.4f}\n")
        f.write(f"Precision: {precision:.4f}\n")
        f.write(f"Recall   : {recall:.4f}\n")
        f.write(f"F1-score : {f1:.4f}\n")
        f.write(f"Macro F1 : {macro_f1:.4f}\n")
        f.write("\nClassification Report:\n")
        f.write(report)
        f.write("\nConfusion Matrix:\n")
        f.write(str(cm))

    print("\n" + "=" * 60)
    print("Siamese evaluation completed!")

if __name__ == "__main__":
    main()

Writing /content/evaluate_siamese.py


In [ ]:
!python /content/evaluate_siamese.py

Device: cuda
Loaded epoch: 5
Val F1: 0.6167390356759657
Test samples: 982

Best threshold found: 0.48 with Macro F1: 0.6634

SIAMESE RESNET18 TEST RESULTS
Accuracy : 0.5774
Precision: 0.5439
Recall   : 0.8503
F1-score : 0.6634
Macro F1 : 0.5479

Classification Report:
              precision    recall  f1-score   support

        real     0.6870    0.3154    0.4323       501
        fake     0.5439    0.8503    0.6634       481

    accuracy                         0.5774       982
   macro avg     0.6154    0.5828    0.5479       982
weighted avg     0.6169    0.5774    0.5455       982

Confusion Matrix:
[[158 343]
 [ 72 409]]
Figure(640x480)

Siamese evaluation completed!


In [ ]:
BASELINE_PATH = "/content/drive/MyDrive/Deepfake/best_baseline.pth"
SIAMESE_PATH = "/content/drive/MyDrive/Deepfake/best_siamese.pth"
print("Baseline:", BASELINE_PATH)
print("Exists:", os.path.exists(BASELINE_PATH))
print("Siamese:", SIAMESE_PATH)
print("Exists:", os.path.exists(SIAMESE_PATH))

Baseline: /content/drive/MyDrive/Deepfake/best_baseline.pth
Exists: True
Siamese: /content/drive/MyDrive/Deepfake/best_siamese.pth
Exists: True


In [ ]:
import os, re, shutil

BASE_DIR = "/content/drive/MyDrive/Deepfake"
BASELINE_RESULT = os.path.join(BASE_DIR, "evaluation_baseline", "baseline_test_results.txt")
SIAMESE_RESULT = os.path.join(BASE_DIR, "evaluation_siamese", "siamese_test_results.txt")
BEST_MODEL_PATH = os.path.join(BASE_DIR, "best_model.pth")

def read_test_f1(result_path):
    with open(result_path, "r", encoding="utf-8") as f:
        content = f.read()
    # Lấy điểm F1-score (chính là F1 của class Fake - tiêu chuẩn Kaggle/Deepfake)
    match = re.search(r"F1-score\s*:\s*([0-9.]+)", content)
    return float(match.group(1))

baseline_f1 = read_test_f1(BASELINE_RESULT)
siamese_f1 = read_test_f1(SIAMESE_RESULT)

print(f"Baseline ResNet18 F1 : {baseline_f1:.4f}")
print(f"Siamese ResNet18 F1  : {siamese_f1:.4f}")

# Chọn model có F1 cao hơn
if baseline_f1 >= siamese_f1:
    best_model_name, best_model_source, best_f1 = "Baseline", os.path.join(BASE_DIR, "best_baseline.pth"), baseline_f1
else:
    best_model_name, best_model_source, best_f1 = "Siamese", os.path.join(BASE_DIR, "best_siamese.pth"), siamese_f1

# Copy thành best_model.pth cho Backend/Frontend dùng
shutil.copy2(best_model_source, BEST_MODEL_PATH)
print(f"\n=> BEST MODEL: {best_model_name} (F1: {best_f1:.4f})")

Baseline ResNet18 F1 : 0.6120
Siamese ResNet18 F1  : 0.6634

=> BEST MODEL: Siamese (F1: 0.6634)
